# Live Webcam Test

Loads `best_model_mobilenet.pth`, opens the built-in webcam (index 0), runs MediaPipe Hands landmark extraction, feeds the 126-feature vector into the classifier, and overlays the predicted ASL letter live on screen.

**Press `q`** in the OpenCV window to quit.

## 1 · Imports & path setup

In [ ]:
import sys
import pathlib
import time
import collections

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
from torchvision import models
import os
import config

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
print(f"MediaPipe   : {mp.__version__}")
print(f"OpenCV      : {cv2.__version__}")

# ── Project root (one level up from training/) ──────────────────────────────
sys.path.insert(0, os.path.abspath(".."))
PROJECT_ROOT = pathlib.Path(config.BASE_DIR)
MODEL_PATH   = PROJECT_ROOT / "models" / "saved" / "best_model_mobilenet.pth"

assert MODEL_PATH.exists(), f"Model not found at {MODEL_PATH}"
print(f"\nModel path  : {MODEL_PATH}  ({MODEL_PATH.stat().st_size/1e6:.2f} MB)")


Python      : 3.10.20
PyTorch     : 2.5.1+cu121
CUDA avail  : True
MediaPipe   : 0.10.14
OpenCV      : 4.9.0

Model path  : C:\Not Windows\Artificial Intelligence and Machine Learning (AIM)\Summer 2026\Capstone Project\models\saved\best_model_mobilenet.pth  (11.83 MB)


## 2 · Class labels (must match training order)

In [2]:
# 29 classes — alphabetical order matches torchvision ImageFolder default
CLASS_NAMES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z',
    'del','nothing','space'
]
assert len(CLASS_NAMES) == 29
print("Classes:", CLASS_NAMES)

Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


## 3 · Rebuild model architecture & load weights

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Architecture must match previous step exactly ─────────────────────────────────
model = models.mobilenet_v2(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.2),        # 0
    nn.Linear(1280, 512),   # 1
    nn.ReLU(),              # 2
    nn.Dropout(0.2),        # 3
    nn.Linear(512, 29),     # 4
)

state = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(state)
model.to(DEVICE)
model.eval()
print("Model loaded and set to eval mode ✅")

Device: cuda
Model loaded and set to eval mode ✅


## 4 · Landmark helper functions

Replicates the exact same preprocessing pipeline used in Steps 7–9 so the feature vector is identical to what the model was trained on.

In [4]:
from torchvision import transforms
from PIL import Image

# Matches val_transform from train_mobilenet.ipynb exactly
inference_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def preprocess_frame(frame_crop: np.ndarray) -> torch.Tensor:
    """Convert a BGR numpy crop → (1, 3, 128, 128) tensor."""
    rgb = cv2.cvtColor(frame_crop, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    tensor = inference_transform(pil).unsqueeze(0).to(DEVICE)
    return tensor

print("Preprocessing defined ✅")

Preprocessing defined ✅


> **Note — preprocessing alignment:** If your Step 7–9 pipeline used a different method to reach shape `(126,)` (e.g. a saved `StandardScaler`, or a different distance formula), swap out `extract_landmarks()` above to match exactly. The model will silently misclassify if the feature construction differs.

## 5 · Inference helper

In [5]:
def predict(frame_crop: np.ndarray) -> tuple[str, float]:
    """Takes a BGR numpy crop, returns (class_name, confidence)."""
    tensor = preprocess_frame(frame_crop)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)
        conf, idx = probs.max(dim=1)
    return CLASS_NAMES[idx.item()], conf.item()

print("Inference helper defined ✅")

Inference helper defined ✅


## 6 · Quick sanity-check (no webcam needed)

Runs a single forward pass with a zero-vector to confirm the model loads and outputs 29 logits correctly.

In [6]:
import numpy as np
dummy_crop = np.zeros((128, 128, 3), dtype=np.uint8)
lbl, conf = predict(dummy_crop)
print(f"Sanity check → predicted: '{lbl}'  confidence: {conf*100:.2f}%")
print("Shape check  → 128×128 image in, 29 classes out ✅")

Sanity check → predicted: 'nothing'  confidence: 86.69%
Shape check  → 128×128 image in, 29 classes out ✅


## 7 · Live webcam loop

- Built-in camera → index `0`  
- **Press `q`** to quit cleanly  
- Prediction smoothed over a 5-frame rolling window to reduce flicker  
- FPS counter displayed top-right

In [7]:
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

CAMERA_INDEX      = 0
CONFIDENCE_THRESH = 0.70
SMOOTH_WINDOW     = 5
PADDING           = 40   # pixels to pad around hand bounding box

FONT        = cv2.FONT_HERSHEY_SIMPLEX
COLOR_GREEN = (0, 255, 100)
COLOR_RED   = (0, 60, 255)
COLOR_WHITE = (255, 255, 255)
COLOR_BLACK = (0, 0, 0)

def get_hand_crop(frame, hand_landmarks, h, w, padding=PADDING):
    """Crop the hand region from the frame with padding."""
    xs = [lm.x * w for lm in hand_landmarks.landmark]
    ys = [lm.y * h for lm in hand_landmarks.landmark]
    x1 = max(0, int(min(xs)) - padding)
    y1 = max(0, int(min(ys)) - padding)
    x2 = min(w, int(max(xs)) + padding)
    y2 = min(h, int(max(ys)) + padding)
    return frame[y1:y2, x1:x2], x1, y1, x2, y2

def draw_label(frame, text, conf, x1, y1):
    color = COLOR_GREEN if conf >= CONFIDENCE_THRESH else COLOR_RED
    label = f"{text}  {conf*100:.1f}%"
    (tw, th), baseline = cv2.getTextSize(label, FONT, 1.2, 2)
    pad = 8
    cv2.rectangle(frame,
                  (x1 - pad, y1 - th - pad * 2),
                  (x1 + tw + pad, y1),
                  COLOR_BLACK, -1)
    cv2.putText(frame, label, (x1, y1 - pad), FONT, 1.2, color, 2, cv2.LINE_AA)

cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("Cannot open camera index 0.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

pred_buffer = collections.deque(maxlen=SMOOTH_WINDOW)
prev_time   = time.time()

print("Webcam open ✅  —  press  q  in the OpenCV window to quit")

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.5,
) as hands:

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Frame grab failed — exiting")
            break

        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = hands.process(rgb)
        rgb.flags.writeable = True

        if results.multi_hand_landmarks:
            for hand_lm in results.multi_hand_landmarks:
                # Draw skeleton
                mp_drawing.draw_landmarks(
                    frame, hand_lm, mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style(),
                )

                # Crop hand region → predict
                crop, x1, y1, x2, y2 = get_hand_crop(frame, hand_lm, h, w)
                if crop.size == 0:
                    continue

                label, conf = predict(crop)
                pred_buffer.append(label)
                smoothed = collections.Counter(pred_buffer).most_common(1)[0][0]

                # Draw bounding box + label
                cv2.rectangle(frame, (x1, y1), (x2, y2), COLOR_GREEN, 2)
                draw_label(frame, smoothed, conf, x1, y1)
        else:
            pred_buffer.clear()
            cv2.putText(frame, "No hand detected", (20, h - 20),
                        FONT, 0.7, COLOR_RED, 2, cv2.LINE_AA)

        now = time.time()
        fps = 1.0 / max(now - prev_time, 1e-6)
        prev_time = now
        cv2.putText(frame, f"FPS: {fps:.1f}", (w - 130, 30),
                    FONT, 0.7, COLOR_WHITE, 2, cv2.LINE_AA)

        cv2.imshow("Signify — Step 11 Live Test  [q = quit]", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Quit signal received.")
            break

cap.release()
cv2.destroyAllWindows()
print("Camera released. Step 11 complete ✅")

Webcam open ✅  —  press  q  in the OpenCV window to quit


c:\Users\eng_b\miniconda3\envs\signify\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Quit signal received.
Camera released. Step 11 complete ✅


---
## Troubleshooting

| Symptom | Fix |
|---|---|
| `RuntimeError: Cannot open camera` | Change `CAMERA_INDEX` to `1` |
| Window opens but all predictions wrong | Check your Step 7–9 preprocessing — swap `extract_landmarks()` to match exactly |
| Low confidence on every sign | Lighting — ensure hand is well-lit against a plain background |
| `CUDA out of memory` | Set `DEVICE = torch.device('cpu')` manually |
| TF deprecation warnings | Harmless — MediaPipe 0.10.x ships with TF internals, ignore |

---
**Next → Step 12:** `agents/vision_agent.py`